# Notebook 1: Simple Movie Chatbot with Memory

## Learning Objectives
- Build a conversational chatbot with memory
- Load configuration from Excel files
- Use model-agnostic LLM loading
- Maintain conversation context across messages

## Features:
✅ **Conversational Memory** - Remembers previous messages  
✅ **Model-Agnostic** - Works with OpenAI, Gemini, Mistral, Claude  
✅ **Excel Configuration** - Loads prompts from file  
✅ **Thread-based Conversations** - Multiple conversation threads

## Installation

In [ ]:
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai --quiet

## Imports

In [1]:
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

from ai_course_utils import load_llm_from_env, load_use_case_config, display_config

print("✓ Imports successful")

✓ Imports successful


## Configuration

In [2]:
load_dotenv()
load_dotenv('../.env')
display_config()

API CONFIGURATION (.env file)
LLM Provider:    openai
LLM Model:       gpt-4.1-mini
Temperature:     0.0

API Keys Status:
  OpenAI               ✓ Set
  Google               ✗ Not set
  Mistral              ✗ Not set
  Anthropic            ✗ Not set
  Serper (Web Search)  ✓ Set

Data Files:
  Provide file paths as function parameters
  Example: load_use_case_config('your_file.xlsx')


## Load Use Case

In [3]:
use_case_file = "../data/use_case_Movie_Recommendation.xlsx"
use_case_config = load_use_case_config(use_case_file)
system_prompt = use_case_config.get("agent_prompt", "You are a helpful movie assistant")
print(f"✓ System prompt loaded")

✓ Use case loaded: ../data/use_case_Movie_Recommendation.xlsx
  Components: user, agent_prompt, url
✓ System prompt loaded


## Initialize LLM

In [4]:
llm = load_llm_from_env()
print("✓ LLM initialized")

🤖 Loading LLM: openai / gpt-4.1-mini
✓ LLM initialized


## Build Chatbot with Memory

In [5]:
def chatbot(state: MessagesState) -> dict:
    """Chatbot with conversation memory."""
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

# Build graph
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

# Compile with memory
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("✓ Chatbot with memory ready")

✓ Chatbot with memory ready


## Chat Interface

In [6]:
def chat(user_input: str, thread_id: str = "default") -> str:
    """
    Chat with the movie assistant.
    
    Args:
        user_input: Your message
        thread_id: Conversation thread (use same ID to continue conversation)
        
    Returns:
        Assistant's response
    """
    config = {"configurable": {"thread_id": thread_id}}
    result = graph.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config
    )
    return result["messages"][-1].content

print("🎬 Movie Chatbot Ready!")
print("\nUsage:")
print("  response = chat('Your message here')")
print("  response = chat('Follow-up question', thread_id='same_thread')")

🎬 Movie Chatbot Ready!

Usage:
  response = chat('Your message here')
  response = chat('Follow-up question', thread_id='same_thread')


## Test Conversations

In [7]:
# Start a new conversation
print("="*70)
print("Conversation 1: Movie Recommendation")
print("="*70)
print("\nUser: Recommend a sci-fi movie")
response = chat("Recommend a sci-fi movie", thread_id="conv1")
print(f"Bot: {response}")

print("\n" + "-"*70)
print("\nUser: What about something with time travel?")
response = chat("What about something with time travel?", thread_id="conv1")
print(f"Bot: {response}")

Conversation 1: Movie Recommendation

User: Recommend a sci-fi movie
Bot: Sure! To help me recommend the best sci-fi movie for you, could you tell me if you prefer something more action-packed, thought-provoking, or maybe a classic? Also, do you have a favorite decade or country of origin for movies?

----------------------------------------------------------------------

User: What about something with time travel?
Bot: Great choice! Time travel movies can be really fun and mind-bending. Do you prefer a more serious and complex story like "Primer," or something with a mix of humor and adventure like "Back to the Future"? Also, are you looking for a recent film or a classic?


In [ ]:
# Start another independent conversation
print("\n" + "="*70)
print("Conversation 2: Family Movies")
print("="*70)
print("\nUser: What's a good family movie?")
response = chat("What's a good family movie?", thread_id="conv2")
print(f"Bot: {response}")

## Interactive Testing

In [8]:
# Try your own conversation
my_thread = "my_conversation"

# First message
query1 = "What is frequency distribution of movies by country by year in the last 5 years "
print(f"User: {query1}")
response1 = chat(query1, thread_id=my_thread)
print(f"Bot: {response1}")

# Follow-up (will remember context)
query2 = "What is count of movies by language in the last 5 years"
print(f"\nUser: {query2}")
response2 = chat(query2, thread_id=my_thread)
print(f"Bot: {response2}")

User: What is frequency distribution of movies by country by year in the last 5 years 
Bot: That’s an interesting data request! To clarify, are you looking for a general overview of how many movies were produced by different countries each year over the last 5 years? Or are you interested in specific genres or types of movies? Also, do you want this information for a particular region or globally? This will help me provide the most relevant information.

User: What is count of movies by language in the last 5 years
Bot: Got it! Are you interested in the count of movies by language globally, or focused on a specific region or country? Also, do you want this data for all genres or specific ones? This will help me tailor the information better for you.


In [ ]:
# Try your own conversation
my_thread = "my_conversation"

# First message
query1 = "I love action movies"
print(f"User: {query1}")
response1 = chat(query1, thread_id=my_thread)
print(f"Bot: {response1}")

# Follow-up (will remember context)
query2 = "Which one has the best fight scenes?"
print(f"\nUser: {query2}")
response2 = chat(query2, thread_id=my_thread)
print(f"Bot: {response2}")